# Gas Distribution Analysis

This notebook provides a **detailed analysis of gas distribution** in nuclear fuel during irradiation. Understanding how fission gas atoms distribute themselves among different reservoirs is crucial for predicting swelling and gas release behavior.

## Learning Objectives

By the end of this notebook, you will:
- Understand the **five main gas reservoirs** in the model
- Learn how to **track gas distribution** over time
- Visualize gas **redistribution during irradiation**
- Compare gas behavior in **bulk vs phase boundaries**
- Analyze **gas release mechanisms** and timing
- Understand the relationship between **distribution and swelling**

## Physical Background

During nuclear fission, gas atoms (Xe, Kr) are produced and distribute themselves among several reservoirs:

1. **Bulk Solution**: Gas atoms dissolved in the grain interior matrix
2. **Bulk Bubbles**: Gas atoms trapped in bubbles within grain interiors
3. **Boundary Solution**: Gas atoms at grain boundaries/phase interfaces
4. **Boundary Bubbles**: Gas atoms in bubbles at interfaces
5. **Released Gas**: Gas that has escaped the fuel material

The distribution evolves over time as:
- Gas is continuously produced by fission
- Gas atoms diffuse through the material
- Bubbles nucleate, grow, and may interconnect
- Gas is released when bubbles become interconnected

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters
from gas_swelling.visualization.comparison_plots import (
    plot_gas_distribution_pie,
    plot_gas_distribution_pie_simple,
    plot_gas_distribution_evolution,
    calculate_gas_distribution_analysis,
    compare_bulk_interface
)

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to analyze gas distribution in nuclear fuel.")

## 1. Understanding Gas Reservoirs

Let's start by running a simulation and examining the final gas distribution.

In [ ]:
# Create default parameters and run simulation
params = create_default_parameters()
model = GasSwellingModel(params)

# Simulation time: 100 days
sim_time_days = 100
sim_time_sec = sim_time_days * 24 * 3600
t_eval = np.linspace(0, sim_time_sec, 100)

print(f"Running simulation for {sim_time_days} days at {params['temperature']} K...")
print("(This may take 10-30 seconds...)")

result = model.solve(
    t_span=(0, sim_time_sec),
    t_eval=t_eval
)

print("\n✅ Simulation completed successfully!")

### 1.1 Detailed Gas Distribution (5 Reservoirs)

Let's visualize the gas distribution across all five reservoirs at the end of the simulation.

In [ ]:
# Create detailed pie chart showing all 5 reservoirs
fig = plot_gas_distribution_pie(result, time_index=-1)
plt.show()

# Calculate and print numerical values
Cgb_final = result['Cgb'][-1]
Cgf_final = result['Cgf'][-1]
Ccb_final = result['Ccb'][-1]
Ccf_final = result['Ccf'][-1]
Ncb_final = result['Ncb'][-1]
Ncf_final = result['Ncf'][-1]
released_final = result['released_gas'][-1]

# Gas in each reservoir
gas_bulk_solution = Cgb_final
gas_bulk_bubbles = Ccb_final * Ncb_final
gas_boundary_solution = Cgf_final
gas_boundary_bubbles = Ccf_final * Ncf_final
gas_released = released_final

total_gas = gas_bulk_solution + gas_bulk_bubbles + gas_boundary_solution + gas_boundary_bubbles + gas_released

print("=" * 70)
print("GAS DISTRIBUTION AT FINAL TIME")
print("=" * 70)
print(f"\n📍 BULK (grain interior):")
print(f"   In solution: {gas_bulk_solution/total_gas*100:.2f}%")
print(f"   In bubbles:  {gas_bulk_bubbles/total_gas*100:.2f}%")
print(f"   Bulk total:  {(gas_bulk_solution + gas_bulk_bubbles)/total_gas*100:.2f}%")

print(f"\n📍 BOUNDARIES (grain boundaries/interfaces):")
print(f"   In solution: {gas_boundary_solution/total_gas*100:.2f}%")
print(f"   In bubbles:  {gas_boundary_bubbles/total_gas*100:.2f}%")
print(f"   Boundary total: {(gas_boundary_solution + gas_boundary_bubbles)/total_gas*100:.2f}%")

print(f"\n💨 RELEASED:")
print(f"   Released:   {gas_released/total_gas*100:.2f}%")

print(f"\n📊 ABSOLUTE VALUES:")
print(f"   Bulk solution:    {gas_bulk_solution:.3e} atoms/m³")
print(f"   Bulk bubbles:     {gas_bulk_bubbles:.3e} atoms/m³")
print(f"   Boundary solution: {gas_boundary_solution:.3e} atoms/m³")
print(f"   Boundary bubbles:  {gas_boundary_bubbles:.3e} atoms/m³")
print(f"   Released:         {gas_released:.3e} atoms/m³")
print(f"   Total:            {total_gas:.3e} atoms/m³")

### 1.2 Simplified Gas Distribution (3 Categories)

Often we group gas into three broad categories for simpler analysis:
- **Solution**: Gas dissolved in matrix (bulk + boundary)
- **In Bubbles**: Gas trapped in bubbles (bulk + boundary)
- **Released**: Gas that has escaped the fuel

In [ ]:
# Create simplified pie chart
fig = plot_gas_distribution_pie_simple(result, time_index=-1)
plt.show()

# Calculate simplified distribution
analysis = calculate_gas_distribution_analysis(result, time_index=-1)

print("=" * 70)
print("SIMPLIFIED GAS DISTRIBUTION")
print("=" * 70)
print(f"\n💧 In Solution: {analysis['gas_in_solution_fraction']*100:.2f}%")
print(f"   → Gas atoms dissolved in the material matrix")
print(f"   → Can diffuse and contribute to bubble growth")

print(f"\n🫧 In Bubbles: {analysis['gas_in_bubbles_fraction']*100:.2f}%")
print(f"   → Gas atoms trapped in bubbles")
print(f"   → Contributes to fuel swelling")
print(f"   → May be released if bubbles interconnect")

print(f"\n💨 Released: {analysis['gas_release_fraction']*100:.2f}%")
print(f"   → Gas that has escaped the fuel")
print(f"   → No longer contributes to swelling")
print(f"   → Important for fuel performance and safety")

## 2. Gas Distribution Evolution Over Time

Gas distribution changes continuously during irradiation. Let's visualize this evolution.

In [ ]:
# Plot gas distribution evolution as stacked area plot
time_days = result['time'] / (24 * 3600)
fig = plot_gas_distribution_evolution(result, time_unit='days')
plt.show()

print("\n🔍 EVOLUTION STAGES:")
print("   1. Early time: Most gas is in solution (diffusing through matrix)")
print("   2. Intermediate: Gas accumulates in bubbles (nucleation and growth)")
print("   3. Late time: Significant gas may be released (bubble interconnection)")

### 2.1 Time Evolution of Each Reservoir

Let's examine how each individual gas reservoir evolves over time.

In [ ]:
# Calculate gas in each reservoir over time
gas_bulk_solution = result['Cgb']
gas_bulk_bubbles = result['Ccb'] * result['Ncb']
gas_boundary_solution = result['Cgf']
gas_boundary_bubbles = result['Ccf'] * result['Ncf']
gas_released = result['released_gas']

# Create multi-panel plot
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Gas Reservoir Evolution Over Time', fontsize=14, fontweight='bold')

# Plot 1: Bulk solution
axes[0, 0].plot(time_days, gas_bulk_solution, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Time (days)')
axes[0, 0].set_ylabel('Gas (atoms/m³)')
axes[0, 0].set_title('Bulk Solution')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Bulk bubbles
axes[0, 1].plot(time_days, gas_bulk_bubbles, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Time (days)')
axes[0, 1].set_ylabel('Gas (atoms/m³)')
axes[0, 1].set_title('Bulk Bubbles')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Boundary solution
axes[0, 2].plot(time_days, gas_boundary_solution, 'r-', linewidth=2)
axes[0, 2].set_xlabel('Time (days)')
axes[0, 2].set_ylabel('Gas (atoms/m³)')
axes[0, 2].set_title('Boundary Solution')
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Boundary bubbles
axes[1, 0].plot(time_days, gas_boundary_bubbles, 'orange', linewidth=2)
axes[1, 0].set_xlabel('Time (days)')
axes[1, 0].set_ylabel('Gas (atoms/m³)')
axes[1, 0].set_title('Boundary Bubbles')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Released gas
axes[1, 1].plot(time_days, gas_released, 'purple', linewidth=2)
axes[1, 1].set_xlabel('Time (days)')
axes[1, 1].set_ylabel('Gas (atoms/m³)')
axes[1, 1].set_title('Released Gas')
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Simplified categories
total_gas_time = gas_bulk_solution + gas_bulk_bubbles + gas_boundary_solution + gas_boundary_bubbles + gas_released
solution_fraction = (gas_bulk_solution + gas_boundary_solution) / total_gas_time * 100
bubble_fraction = (gas_bulk_bubbles + gas_boundary_bubbles) / total_gas_time * 100
released_fraction = gas_released / total_gas_time * 100

axes[1, 2].plot(time_days, solution_fraction, label='Solution', linewidth=2)
axes[1, 2].plot(time_days, bubble_fraction, label='Bubbles', linewidth=2)
axes[1, 2].plot(time_days, released_fraction, label='Released', linewidth=2)
axes[1, 2].set_xlabel('Time (days)')
axes[1, 2].set_ylabel('Fraction (%)')
axes[1, 2].set_title('Simplified Categories')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Bulk vs Boundary Gas Distribution

One key aspect of the model is tracking gas behavior in two locations:
- **Bulk**: Grain interiors (away from boundaries)
- **Boundaries**: Grain boundaries and phase interfaces

These locations have different behaviors due to:
- Different defect concentrations
- Different sink strengths
- Different bubble nucleation rates

In [ ]:
# Compare bulk vs boundary behavior
fig = compare_bulk_interface(result, variables=['Cg', 'Cc', 'Nc'], 
                             time_unit='days', figsize=(14, 10))
plt.show()

print("\n🔍 BULK vs BOUNDARY COMPARISON:")
print(f"\nGas concentration (Cg):")
print(f"   Bulk:   {result['Cgb'][-1]:.3e} atoms/m³")
print(f"   Boundary: {result['Cgf'][-1]:.3e} atoms/m³")
print(f"   Ratio:   {result['Cgf'][-1]/result['Cgb'][-1] if result['Cgb'][-1] > 0 else 0:.2f}x")

print(f"\nBubble concentration (Cc):")
print(f"   Bulk:   {result['Ccb'][-1]:.3e} bubbles/m³")
print(f"   Boundary: {result['Ccf'][-1]:.3e} bubbles/m³")
print(f"   Ratio:   {result['Ccf'][-1]/result['Ccb'][-1] if result['Ccb'][-1] > 0 else 0:.2f}x")

print(f"\nGas atoms per bubble (Nc):")
print(f"   Bulk:   {result['Ncb'][-1]:.0f} atoms/bubble")
print(f"   Boundary: {result['Ncf'][-1]:.0f} atoms/bubble")
print(f"   Ratio:   {result['Ncf'][-1]/result['Ncb'][-1] if result['Ncb'][-1] > 0 else 0:.2f}x")

## 4. Gas Distribution and Swelling

The primary concern with gas distribution is its impact on fuel swelling. Let's analyze how gas distribution relates to swelling.

In [ ]:
# Calculate swelling over time
V_bubbles_bulk = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
V_bubbles_boundary = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
swelling = (V_bubbles_bulk + V_bubbles_boundary) * 100

# Create figure comparing gas distribution and swelling
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Gas distribution evolution
axes[0].stackplot(time_days, 
                  solution_fraction,
                  bubble_fraction, 
                  released_fraction,
                  labels=['Solution', 'Bubbles', 'Released'],
                  alpha=0.8)
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Gas Fraction (%)')
axes[0].set_title('Gas Distribution Evolution')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# Plot 2: Swelling evolution
axes[1].plot(time_days, swelling, 'r-', linewidth=2, label='Total Swelling')
axes[1].fill_between(time_days, swelling, alpha=0.3, color='red')
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Swelling (%)')
axes[1].set_title('Fuel Swelling Evolution')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n🔍 RELATIONSHIP BETWEEN GAS DISTRIBUTION AND SWELLING:")
print(f"\nFinal swelling: {swelling[-1]:.4f}%")
print(f"   → Gas in bubbles: {bubble_fraction[-1]:.1f}%")
print(f"   → More gas in bubbles → larger bubbles → more swelling")

## 5. Temperature Effects on Gas Distribution

Temperature strongly affects gas distribution. Let's compare different temperatures.

In [ ]:
# Run simulations at different temperatures
temperatures = [600, 700, 800, 900]
print(f"Running simulations at {len(temperatures)} temperatures...")
print("(This may take 30-60 seconds...)")

results_temp = {}
distributions_temp = {}

for temp in temperatures:
    print(f"  Running at {temp} K...", end=" ")
    
    params_temp = create_default_parameters()
    params_temp['temperature'] = temp
    
    model_temp = GasSwellingModel(params_temp)
    result_temp = model_temp.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    results_temp[temp] = result_temp
    distributions_temp[temp] = calculate_gas_distribution_analysis(result_temp, time_index=-1)
    
    print(f"✓")

print("\n✅ All simulations completed!")

In [ ]:
# Compare final gas distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Pie charts at different temperatures
fig_pie, axes_pie = plt.subplots(2, 2, figsize=(12, 12))
axes_pie = axes_pie.flatten()

for idx, temp in enumerate(temperatures):
    result_t = results_temp[temp]
    analysis_t = distributions_temp[temp]
    
    # Pie chart
    sizes = [
        analysis_t['gas_in_solution_fraction'] * 100,
        analysis_t['gas_in_bubbles_fraction'] * 100,
        analysis_t['gas_release_fraction'] * 100
    ]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    axes_pie[idx].pie(sizes, labels=['Solution', 'Bubbles', 'Released'],
                     autopct='%1.1f%%', startangle=90, colors=colors)
    axes_pie[idx].set_title(f'{temp} K', fontweight='bold')

fig_pie.suptitle('Gas Distribution at Different Temperatures', 
                 fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot 2: Quantitative comparison
solution_fractions = [distributions_temp[t]['gas_in_solution_fraction'] * 100 
                      for t in temperatures]
bubble_fractions = [distributions_temp[t]['gas_in_bubbles_fraction'] * 100 
                     for t in temperatures]
released_fractions = [distributions_temp[t]['gas_release_fraction'] * 100 
                      for t in temperatures]

x = np.arange(len(temperatures))
width = 0.25

axes[0].bar(x - width, solution_fractions, width, label='Solution', color='#3498db')
axes[0].bar(x, bubble_fractions, width, label='Bubbles', color='#e74c3c')
axes[0].bar(x + width, released_fractions, width, label='Released', color='#2ecc71')

axes[0].set_xlabel('Temperature (K)')
axes[0].set_ylabel('Gas Fraction (%)')
axes[0].set_title('Gas Distribution vs Temperature')
axes[0].set_xticks(x)
axes[0].set_xticklabels(temperatures)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 3: Swelling vs temperature
swellings = []
for temp in temperatures:
    r = results_temp[temp]
    Vb = (4/3) * np.pi * r['Rcb']**3 * r['Ccb']
    Vf = (4/3) * np.pi * r['Rcf']**3 * r['Ccf']
    swellings.append((Vb + Vf)[-1] * 100)

axes[1].plot(temperatures, swellings, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Temperature (K)')
axes[1].set_ylabel('Final Swelling (%)')
axes[1].set_title('Swelling vs Temperature')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 TEMPERATURE EFFECTS:")
print("   Low T (600 K): Gas mainly in solution (slow diffusion)")
print("   Medium T (700-800 K): Optimal for bubble growth and swelling")
print("   High T (900 K): More gas release, reduced swelling")

## 6. Gas Release Analysis

Gas release is a critical concern for fuel performance. Let's analyze when and how gas is released.

In [ ]:
# Analyze gas release in detail
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Gas Release Analysis', fontsize=14, fontweight='bold')

# Plot 1: Cumulative gas release
axes[0, 0].plot(time_days, result['released_gas'], 'purple', linewidth=2)
axes[0, 0].fill_between(time_days, result['released_gas'], alpha=0.3, color='purple')
axes[0, 0].set_xlabel('Time (days)')
axes[0, 0].set_ylabel('Released Gas (atoms/m³)')
axes[0, 0].set_title('Cumulative Gas Release')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Gas release fraction
total_gas_time = gas_bulk_solution + gas_bulk_bubbles + gas_boundary_solution + gas_boundary_bubbles + gas_released
release_fraction = gas_released / total_gas_time * 100

axes[0, 1].plot(time_days, release_fraction, 'darkred', linewidth=2)
axes[0, 1].set_xlabel('Time (days)')
axes[0, 1].set_ylabel('Gas Release Fraction (%)')
axes[0, 1].set_title('Gas Release Fraction Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Gas release rate (derivative)
release_rate = np.gradient(result['released_gas'], result['time'])
axes[1, 0].semilogy(time_days, release_rate * (24*3600), 'brown', linewidth=2)
axes[1, 0].set_xlabel('Time (days)')
axes[1, 0].set_ylabel('Release Rate (atoms/m³/day)')
axes[1, 0].set_title('Gas Release Rate')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Bubble interconnection indicator
# Boundary bubble radius compared to critical radius
Rcf_nm = result['Rcf'] * 1e9
axes[1, 1].plot(time_days, Rcf_nm, 'blue', linewidth=2, label='Boundary Bubble Radius')
axes[1, 1].axhline(y=10, color='red', linestyle='--', 
                   label='~Interconnection Threshold (~10 nm)')
axes[1, 1].set_xlabel('Time (days)')
axes[1, 1].set_ylabel('Radius (nm)')
axes[1, 1].set_title('Boundary Bubble Growth')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 GAS RELEASE MECHANISM:")
print("   1. Gas atoms diffuse to boundaries")
print("   2. Bubbles grow at boundaries")
print("   3. When bubbles interconnect (radius > ~10 nm), gas can escape")
print("   4. Release rate increases with bubble interconnection")

print(f"\n📊 FINAL VALUES:")
print(f"   Total gas released: {result['released_gas'][-1]:.3e} atoms/m³")
print(f"   Release fraction: {release_fraction[-1]:.2f}%")
print(f"   Final boundary bubble radius: {Rcf_nm[-1]:.2f} nm")

## 7. Practical Analysis: Custom Functions

Let's create some practical functions for gas distribution analysis that you can reuse in your own research.

In [ ]:
def analyze_gas_at_time(result, time_days_target, time_array_days):
    """
    Analyze gas distribution at a specific time point.
    
    Parameters:
    -----------
    result : dict
        Simulation result dictionary
    time_days_target : float
        Target time in days
    time_array_days : array
        Time array in days (same units as target)
    
    Returns:
    --------
    dict : Gas distribution analysis
    """
    # Find closest time index
    idx = np.argmin(np.abs(time_array_days - time_days_target))
    actual_time = time_array_days[idx]
    
    # Get distribution
    analysis = calculate_gas_distribution_analysis(result, time_index=idx)
    
    print(f"\nGas Distribution at t = {actual_time:.1f} days:")
    print(f"  Solution: {analysis['gas_in_solution_fraction']*100:.2f}%")
    print(f"  Bubbles:  {analysis['gas_in_bubbles_fraction']*100:.2f}%")
    print(f"  Released: {analysis['gas_release_fraction']*100:.2f}%")
    
    return analysis


def compare_gas_distributions(result1, result2, label1, label2, time_array_days):
    """
    Compare gas distributions between two simulations.
    
    Parameters:
    -----------
    result1, result2 : dict
        Simulation result dictionaries
    label1, label2 : str
        Labels for the two simulations
    time_array_days : array
        Time array in days
    
    Returns:
    --------
    fig : matplotlib figure
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for result, label in [(result1, label1), (result2, label2)]:
        gas_bulk_sol = result['Cgb']
        gas_bulk_bub = result['Ccb'] * result['Ncb']
        gas_bound_sol = result['Cgf']
        gas_bound_bub = result['Ccf'] * result['Ncf']
        gas_rel = result['released_gas']
        
        total = gas_bulk_sol + gas_bulk_bub + gas_bound_sol + gas_bound_bub + gas_rel
        
        sol_frac = (gas_bulk_sol + gas_bound_sol) / total * 100
        bub_frac = (gas_bulk_bub + gas_bound_bub) / total * 100
        rel_frac = gas_rel / total * 100
        
        ax.plot(time_array_days, sol_frac, label=f'{label} - Solution', linewidth=2)
        ax.plot(time_array_days, bub_frac, label=f'{label} - Bubbles', linewidth=2, linestyle='--')
        ax.plot(time_array_days, rel_frac, label=f'{label} - Released', linewidth=2, linestyle=':')
    
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Gas Fraction (%)')
    ax.set_title('Gas Distribution Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return fig


def find_peak_release_time(result, time_array_days):
    """
    Find the time of maximum gas release rate.
    
    Parameters:
    -----------
    result : dict
        Simulation result dictionary
    time_array_days : array
        Time array in days
    
    Returns:
    --------
    float : Time of peak release rate (days)
    """
    release_rate = np.gradient(result['released_gas'], result['time'])
    peak_idx = np.argmax(release_rate)
    return time_array_days[peak_idx]

# Test the functions
print("\n" + "="*70)
print("TESTING CUSTOM ANALYSIS FUNCTIONS")
print("="*70)

print("\n1. Analyze gas at specific times:")
analyze_gas_at_time(result, 10, time_days)
analyze_gas_at_time(result, 50, time_days)
analyze_gas_at_time(result, 100, time_days)

print(f"\n2. Peak release time:")
peak_time = find_peak_release_time(result, time_days)
print(f"   Maximum release rate occurs at t = {peak_time:.1f} days")

## 8. Summary and Key Insights

### Gas Distribution Fundamentals

1. **Five Reservoirs**:
   - Bulk solution + Bulk bubbles
   - Boundary solution + Boundary bubbles
   - Released gas

2. **Evolution Stages**:
   - **Early**: Gas mainly in solution (diffusing)
   - **Middle**: Gas accumulates in bubbles (nucleation + growth)
   - **Late**: Gas release (bubble interconnection)

3. **Bulk vs Boundary**:
   - Bulk: Generally higher bubble concentration
   - Boundary: Often larger bubbles, more gas release
   - Both contribute to swelling

4. **Temperature Effects**:
   - Low T: Gas trapped in solution
   - Medium T: Optimal for bubble growth (max swelling)
   - High T: Enhanced gas release

5. **Swelling Correlation**:
   - Swelling ∝ gas in bubbles
   - More gas in bubbles → larger bubbles → more swelling
   - Gas release reduces swelling pressure

### Quick Reference

```python
# Calculate gas distribution
from gas_swelling.visualization.comparison_plots import calculate_gas_distribution_analysis

analysis = calculate_gas_distribution_analysis(result, time_index=-1)
print(f"Solution: {analysis['gas_in_solution_fraction']:.2%}")
print(f"Bubbles: {analysis['gas_in_bubbles_fraction']:.2%}")
print(f"Released: {analysis['gas_release_fraction']:.2%}")

# Create visualizations
from gas_swelling.visualization.comparison_plots import (
    plot_gas_distribution_pie_simple,
    plot_gas_distribution_evolution
)

# Pie chart at final time
fig1 = plot_gas_distribution_pie_simple(result, time_index=-1)

# Evolution over time
fig2 = plot_gas_distribution_evolution(result, time_unit='days')
```

### Next Steps

1. **Explore Other Notebooks**:
   - `01_Basic_Simulation_Walkthrough.ipynb`: Getting started
   - `02_Parameter_Sweep_Studies.ipynb`: Systematic parameter studies
   - `04_Experimental_Data_Comparison.ipynb`: Validation against experiments

2. **Apply to Your Research**:
   - Modify parameters for your material system
   - Compare different compositions
   - Analyze gas distribution at different burnups

3. **Further Analysis**:
   - Study nucleation factor effects on bubble number vs size
   - Investigate gas release thresholds
   - Compare different gas equations of state

---

**🎉 Congratulations!** You've completed the Gas Distribution Analysis notebook.

You now have the skills to:
- Analyze gas distribution across all reservoirs
- Track gas evolution during irradiation
- Understand temperature effects on distribution
- Relate gas distribution to swelling behavior
- Use custom functions for your own analysis

Happy analyzing! 🚀